# 📊 Notebook 04: Data Consolidation

**Objective:** Clean and merge all three data sources (Aqueduct, AQUASTAT, GRACE) into a unified dataset.

**Output:** `data/processed/drought_water_stress.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)

import config
from src.processing.data_cleaner import DataCleaner
from src.processing.data_merger import DataMerger

print('✅ Setup complete')

## 1. Load Processed Data from Each Source

In [ ]:
# Check if processed data exists; if not, run extractors
for path, name in [(config.AQUEDUCT_PROCESSED, 'Aqueduct'), 
                    (config.AQUASTAT_PROCESSED, 'AQUASTAT'),
                    (config.GRACE_PROCESSED, 'GRACE')]:
    if path.exists():
        print(f'✅ {name}: {path}')
    else:
        print(f'❌ {name}: Not found. Running extractor...')
        if name == 'Aqueduct':
            from src.extractors.aqueduct_extractor import AqueductExtractor
            ext = AqueductExtractor()
            ext.download_data('baseline_annual')
            ext.save_processed()
        elif name == 'AQUASTAT':
            from src.extractors.aquastat_extractor import AquastatExtractor
            ext = AquastatExtractor()
            ext.download_data()
            ext.save_processed()
        elif name == 'GRACE':
            from src.extractors.grace_extractor import GraceExtractor
            ext = GraceExtractor()
            ext.download_data()
            ext.save_processed()

In [ ]:
# Load all three datasets
aqueduct_df = pd.read_csv(config.AQUEDUCT_PROCESSED)
aquastat_df = pd.read_csv(config.AQUASTAT_PROCESSED)
grace_df = pd.read_csv(config.GRACE_PROCESSED)

print(f'Aqueduct: {aqueduct_df.shape} — {list(aqueduct_df.columns)}')
print(f'AQUASTAT:  {aquastat_df.shape} — {list(aquastat_df.columns)}')
print(f'GRACE:     {grace_df.shape} — {list(grace_df.columns)}')

## 2. Data Cleaning

In [ ]:
cleaner = DataCleaner()

aqueduct_clean = cleaner.clean_aqueduct(aqueduct_df)
cleaner.validate_data(aqueduct_clean, 'Aqueduct (cleaned)')

aquastat_clean = cleaner.clean_aquastat(aquastat_df)
cleaner.validate_data(aquastat_clean, 'AQUASTAT (cleaned)')

grace_clean = cleaner.clean_grace(grace_df)
cleaner.validate_data(grace_clean, 'GRACE (cleaned)')

## 3. Data Merging

In [ ]:
merger = DataMerger()
consolidated = merger.merge_all(aqueduct_clean, aquastat_clean, grace_clean)

print(f'\n📊 Consolidated Dataset:')
print(f'   Shape: {consolidated.shape}')
print(f'   Columns: {list(consolidated.columns)}')
consolidated.head(10)

In [ ]:
# Summary
summary = merger.get_summary(consolidated)
for k, v in summary.items():
    print(f'{k}: {v}')

In [ ]:
# Data quality check
cleaner.validate_data(consolidated, 'Consolidated')

## 4. Save Consolidated Dataset

In [ ]:
output = merger.save_consolidated(consolidated)
print(f'\n✅ Consolidated dataset saved to: {output}')
print(f'   Total records: {len(consolidated):,}')
print(f'   Total features: {len(consolidated.columns)}')
print(f'   File size: {output.stat().st_size / 1024 / 1024:.1f} MB')